In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = 4  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    "4b": ["lpips==0.1.4", "pycolmap==4.2.1", "ninja"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1", "open3d==0.20.0"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {"4b": ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"],
                      6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}
# Reference objects with a course-made TRELLIS.2 archive, generated-<object>.tar.gz.
GENERATED_OBJECTS = ("boxed_beverage_001", "hammer_014")
# The TRELLIS.2 model revision those archives were generated with (model_revision in trellis2.json).
GENERATED_MODEL_REVISION = "af44b45f2e35a493886929c6d786e563ec68364d"


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def prev_folder(stage):
    """The folder a stage reads: capture/ for stage 1, stage4/ for stage 4b, stage<N-1>/ otherwise.

    Stage 5 reads stage4/ too: stage 4b refines the maps but leaves the mesh to stage 4.
    """
    if stage == 1:
        return "capture"
    if stage == "4b":
        return "stage4"
    return f"stage{stage - 1}"


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (prev_folder(stage)), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    if not (PROJECT / prev_folder(stage) / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None, dest=None):
    """Download a reference archive and move its top-level folders (or only `names`) into dest.

    dest defaults to PROJECT. A non-empty folder of yours that a reference folder replaces is
    kept as <name>.yours (an older .yours is replaced); ground_truth/ is replaced without a copy.
    Nothing moves until the archive is fully extracted.
    """
    dest = PROJECT if dest is None else Path(dest)
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = dest / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = dest / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = dest / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = dest / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"{prev_folder(stage)}.tar.gz"
    print(f"{prev_folder(stage)}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = prev_folder(stage)
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = "stage0.tar.gz" if stage == 1 else f"{prev}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


def ensure_generated(obj):
    """Fetch the course's TRELLIS.2 archive for a reference object and return its folder.

    generated-<obj>.tar.gz holds that object's stage folders, ground_truth/ and trellis2/. They
    go to generated/<obj>/, which is replaced whole on each download, so your own stage folders
    are never touched. A failed download leaves the previous generated/<obj>/ as it was. When
    generated/<obj>/trellis2/trellis2.json is already there with GENERATED_MODEL_REVISION, the
    folder is kept and nothing is downloaded.
    """
    if obj not in GENERATED_OBJECTS:
        raise ValueError(f"no generated archive for {obj!r}; the course ships {', '.join(GENERATED_OBJECTS)}")
    out = PROJECT / "generated" / obj
    try:
        revision = json.loads((out / "trellis2" / "trellis2.json").read_text()).get("model_revision")
    except (OSError, ValueError, AttributeError):
        revision = None
    if revision == GENERATED_MODEL_REVISION:
        print(f"generated/{obj}/ is already here (TRELLIS.2 revision {revision[:7]}), so it is not downloaded again.")
        return out
    with atomic_dir(out) as tmp:
        _extract_reference(f"generated-{obj}.tar.gz", dest=tmp)
    return out


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you paint the stage 3 mesh. You cut its surface into flat pieces that tile a square texture, predict the material of the object in 30 photos, and bake those predictions into three texture maps: the base color (albedo), the roughness and the metallic value of every point on the surface. Stage 6 renders the textured mesh under a lamp you did not photograph it under, so the maps must describe the surface alone, with the light of your capture taken out.

A photo does not separate the two. A pixel's brightness is the product of how much light reaches the surface and how much of it the surface reflects, and many pairs give the same product. You split them with Marigold-IID appearance (Ke et al., "Marigold: Affordable Adaptation of Diffusion-Based Image Generators for Image Analysis", 2025), a latent diffusion model fine-tuned from Stable Diffusion 2 that predicts albedo, roughness and metallic from a single image. Its weights `prs-eth/marigold-iid-appearance-v1-1` are released under the CreativeML Open RAIL++-M license, which permits this use and attaches use restrictions. The UV charts come from xatlas (MIT license), and the occlusion test and the rasterization of the charts use Open3D (MIT license).

Runtime budget on a Colab T4: about 4 minutes for 30 frames once the session is set up. The first run in a new session takes about 5 minutes, since it also downloads the Marigold-IID weights, about 2.6 GB.


In [ ]:
import json
import time

import cv2
import numpy as np

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
STAGE1, STAGE2, STAGE3 = PROJECT / "stage1", PROJECT / "stage2", PROJECT / "stage3"
REFERENCE = on_reference(STAGE)  # the reference stage3/ carries a .reference marker
frames = sorted((STAGE1 / "frames").glob("*.jpg"))
heldout = set((STAGE1 / "heldout.txt").read_text().split())
drifting = set(json.loads((STAGE2 / "drift.json").read_text())["flagged"])
train = [p for p in frames if p.name not in heldout and p.name not in drifting]
N_VIEWS = 30
chosen = [train[i] for i in np.unique(np.linspace(0, len(train) - 1, N_VIEWS).round().astype(int))]
masks = {p.name: cv2.imread(str(STAGE2 / "masks" / (p.stem + ".png")), cv2.IMREAD_GRAYSCALE) > 127 for p in chosen}
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print(f"{len(frames)} frames, {len(train)} usable for training; baking from {len(chosen)} of them, "
      f"{chosen[0].name} to {chosen[-1].name}")

## Cutting the surface into charts

A texture is a function on the unit square, and a mesh is a surface in space. A UV map links them: every vertex gets a coordinate $(u, v) \in [0, 1]^2$, every triangle maps to a triangle in the square, and a point inside a triangle takes the texture value at the matching point of its image. A closed surface such as your object cannot be flattened onto the plane in one piece without tearing, so the map cuts it into *charts*. Each chart is a patch of triangles flattened with little stretch, and the charts are packed into the square with a gap of a few texels between them. Along a cut, the same 3D vertex appears once in each chart it borders, with a different $(u, v)$ in each.

The stage 3 mesh has about 300,000 triangles, more than one texture needs. A triangle smaller than a texel gets no texel of its own, and every extra triangle adds chart boundary that costs packing space. The cell below first decimates the mesh to 50,000 triangles by quadric edge collapse, which removes the edges whose removal moves the surface least. It prints the Chamfer distance between the decimated and the full mesh next to the distance between two samplings of the full mesh, which is the smallest value the measurement can show. Stage 5 still computes volume and inertia from the full mesh.

The packing then sets one scale for every chart, in texels per meter, so that a texel covers the same area everywhere on the surface. The cell prints that size in millimeters.


In [ ]:
import numpy as np
import open3d as o3d
from scipy.spatial import cKDTree


def mesh_chamfer_mm(a, b, n: int = 200_000, seed: int = 0) -> float:
    """Symmetric Chamfer distance in millimeters between two Open3D meshes in meters."""
    o3d.utility.random.seed(seed)
    pa = np.asarray(a.sample_points_uniformly(n).points)
    pb = np.asarray(b.sample_points_uniformly(n).points)
    return float(500 * (cKDTree(pb).query(pa)[0].mean() + cKDTree(pa).query(pb)[0].mean()))


def unwrap(mesh, target_triangles: int = 50_000, padding: int = 2) -> dict:
    """Decimate the mesh and cut it into UV charts with xatlas.

    Returns mesh_uv, a dict of numpy arrays: "vertices" (V x 3, meters), "faces" (F x 3),
    "uvs" (V x 2 in [0, 1], origin at the top-left of the texture, v down) and "normals"
    (V x 3, unit). xatlas splits vertices along chart seams, so V is larger than the
    decimated mesh's vertex count, and a seam vertex appears once per chart.
    """
    import xatlas

    small = mesh.simplify_quadric_decimation(target_triangles)
    small.remove_unreferenced_vertices()
    small.compute_vertex_normals()
    vertices = np.asarray(small.vertices, dtype=np.float32)
    faces = np.asarray(small.triangles, dtype=np.uint32)
    area = small.get_surface_area()
    fill = 0.5  # target share of the texture covered by charts; the rest is padding and gaps
    while True:  # texels per meter so that the charts fill one 2048 page, shrunk until they fit on one
        atlas = xatlas.Atlas()
        atlas.add_mesh(vertices, faces)
        pack = xatlas.PackOptions()
        pack.resolution, pack.padding, pack.bilinear = 2048, padding, True
        pack.texels_per_unit = float(np.sqrt(fill * 2048**2 / area))
        atlas.generate(pack_options=pack)
        if atlas.atlas_count == 1:
            break
        fill *= 0.85
    vmapping, indices, uvs = atlas[0]
    normals = np.asarray(small.vertex_normals)[vmapping]
    return {"vertices": vertices[vmapping].astype(np.float64), "faces": indices.astype(np.int64),
            "uvs": uvs.astype(np.float64), "normals": normals / np.linalg.norm(normals, axis=1, keepdims=True),
            "charts": int(atlas.chart_count), "texel_mm": 1000 / pack.texels_per_unit, "decimated": small}

In [ ]:
mesh = o3d.io.read_triangle_mesh(str(STAGE3 / "mesh.ply"))
t = time.time()
mesh_uv = unwrap(mesh, target_triangles=50_000)
unwrap_s = time.time() - t
decimation_mm = mesh_chamfer_mm(mesh_uv["decimated"], mesh)
floor_mm = mesh_chamfer_mm(mesh, mesh, seed=1)  # two samplings of the same surface: the sampling floor
print(f"stage 3 mesh {len(mesh.triangles):,} triangles, decimated to {len(mesh_uv['faces']):,}: "
      f"Chamfer to the full mesh {decimation_mm:.3f} mm (sampling floor {floor_mm:.3f} mm)")
print(f"{mesh_uv['charts']:,} UV charts, {len(mesh_uv['vertices']):,} vertices after the seam split, "
      f"one texel is {mesh_uv['texel_mm']:.3f} mm on the surface; {unwrap_s:.0f} s")

## Albedo, roughness and metallic from one photo

Model the light leaving a surface point toward the camera as a diffuse term plus a specular one. For the diffuse part, the observed color $I$ at a pixel is

$$
I = A \cdot S,
$$

where $A$ is the albedo, the fraction of light the surface reflects at each wavelength, and $S$ is the shading, the light that reaches the point, which depends on the lamps, on the angle between them and the surface normal, and on shadows. Given $I$ alone, any $A' = A / c$ with $S' = c \, S$ explains the pixel as well. A brown patch is brown paint under white light or white paint under brown light. The specular part depends on two more properties. Roughness sets how widely a highlight spreads, and the metallic value sets whether the surface reflects mostly its own color (metal) or mostly the light's color (paint, paper, plastic).

Marigold-IID appearance resolves the ambiguity with a learned prior. It was fine-tuned on InteriorVerse, a set of synthetic indoor scenes rendered with a physically based BRDF whose materials are described by exactly these three quantities, so for every training image the true albedo, roughness and metallic were known. Given a new photo, the model returns the decomposition that is typical of those renderings. On your object the result is a guess, shaped by which colors are usually paint and which are usually light in that data. The model runs at 768 pixels on its longer side, four denoising steps, one sample per frame, in half precision. Its albedo is in sRGB like the photo, and roughness and metallic are linear values in $[0, 1]$.

You run it on 30 training frames spaced evenly along the orbit, after leaving out the held-out frames of stage 1 and the frames the stage 2 drift check flagged. The cell prints the implied shading $S = Y_{\text{photo}} / Y_{\text{albedo}}$, the ratio of the photo's luminance to the albedo's in linear light, over the object's pixels. A ratio near 1 means the model put the pixel's brightness into the albedo. A ratio well below 1 means it called the surface brighter than the photo shows and blamed the light.


In [ ]:
import cv2
import numpy as np
from pathlib import Path


def opencv_distortion(model: str, params) -> np.ndarray:
    """OpenCV's (k1, k2, p1, p2) for a COLMAP camera model and its parameter list."""
    params = list(params)
    if model in ("PINHOLE", "SIMPLE_PINHOLE"):
        return np.zeros(4)
    if model == "SIMPLE_RADIAL":  # f, cx, cy, k
        return np.array([params[3], 0.0, 0.0, 0.0])
    if model == "RADIAL":  # f, cx, cy, k1, k2
        return np.array([params[3], params[4], 0.0, 0.0])
    if model == "OPENCV":  # fx, fy, cx, cy, k1, k2, p1, p2
        return np.array(params[4:8], dtype=float)
    raise ValueError(f"camera model {model} is not supported here: stage 1 fits SIMPLE_RADIAL; "
                     "re-run stage 1 or add its distortion terms to opencv_distortion")


def load_views(stage1: Path, frames, masks, data_factor: int = 2):
    """Undistorted images and masks at 1/data_factor resolution, with their cameras.

    Each view is a dict with "name", "K" (3 x 3), "viewmat" (world to camera, 4 x 4, OpenCV
    axes), "image" (H x W x 3, uint8 RGB) and "mask" (H x W, bool). stage1/sparse/0 is already
    in meters in the world frame, Z up.
    """
    import pycolmap

    model = pycolmap.Reconstruction(str(Path(stage1) / "sparse" / "0"))
    by_name = {im.name: im for im in model.images.values()}
    views, skipped = [], []
    for p in frames:
        im = by_name.get(Path(p).name)
        if im is None:
            skipped.append(Path(p).name)
            continue
        cam = model.cameras[im.camera_id]
        K = cam.calibration_matrix().copy()
        K[:2] /= data_factor
        dist = opencv_distortion(cam.model.name, cam.params)
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        size = (img.shape[1] // data_factor, img.shape[0] // data_factor)
        img = cv2.undistort(cv2.resize(img, size, interpolation=cv2.INTER_AREA), K, dist)
        m = cv2.resize(masks[Path(p).name].astype(np.float32), size, interpolation=cv2.INTER_AREA)
        viewmat = np.eye(4)
        viewmat[:3] = im.cam_from_world().matrix()
        views.append({"name": Path(p).name, "K": K, "viewmat": viewmat, "image": img,
                      "mask": cv2.undistort(m, K, dist) > 0.5})
    if skipped:  # write_stage1 accepts a model with 80% of the frames registered
        print(f"{len(skipped)} of {len(frames)} frames have no pose in stage1/sparse/0 (stage 1 did not register "
              f"them) and are left out: {', '.join(skipped)}")
    return views

In [ ]:
import diffusers
import huggingface_hub
import torch
import transformers
from diffusers import MarigoldIntrinsicsPipeline
from PIL import Image

for lib in (diffusers, transformers):
    lib.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()
views = load_views(STAGE1, chosen, masks, data_factor=2)
t = time.time()
pipe = MarigoldIntrinsicsPipeline.from_pretrained(
    "prs-eth/marigold-iid-appearance-v1-1", revision="e7280a0a0fc5a0df0b36050882b3d8b77da22fd9",
    variant="fp16", dtype=torch.float16).to("cuda")
pipe.set_progress_bar_config(disable=True)
print(f"Marigold-IID appearance loaded in {time.time() - t:.0f} s (the first run includes the download)")
torch.cuda.reset_peak_memory_stats()
t = time.time()
for v in views:
    out = pipe(Image.fromarray(v["image"]), num_inference_steps=4, ensemble_size=1, processing_resolution=768,
               generator=torch.Generator("cuda").manual_seed(0))
    albedo, material = out.prediction[0], out.prediction[1]  # albedo in sRGB; roughness and metallic in linear
    v["albedo"] = np.clip(albedo, 0, 1).astype(np.float32)
    v["roughness"] = np.clip(material[..., 0], 0, 1).astype(np.float32)
    v["metallic"] = np.clip(material[..., 1], 0, 1).astype(np.float32)
marigold_s = time.time() - t
print(f"{len(views)} views at {views[0]['image'].shape[1]} x {views[0]['image'].shape[0]} in {marigold_s:.0f} s, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**20:,.0f} MiB")
del pipe
torch.cuda.empty_cache()


def luminance(rgb):
    """Relative luminance of sRGB values in [0, 1], computed in linear light."""
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    return lin @ np.array([0.2126, 0.7152, 0.0722])


ratios = []
for v in views:
    inner = cv2.erode(v["mask"].astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
    y_photo, y_albedo = luminance(v["image"][inner] / 255.0), luminance(v["albedo"][inner])
    keep = y_albedo > 0.01
    ratios.append(y_photo[keep] / y_albedo[keep])
ratios = np.concatenate(ratios)
SHADING = {p: float(np.percentile(ratios, p)) for p in (5, 50, 95)}
print(f"implied shading S = photo / albedo (luminance, linear) on the object: median {SHADING[50]:.2f}, "
      f"5th to 95th percentile {SHADING[5]:.2f} to {SHADING[95]:.2f}")
print(f"mean roughness {np.mean([v['roughness'][v['mask']].mean() for v in views]):.2f}, "
      f"mean metallic {np.mean([v['metallic'][v['mask']].mean() for v in views]):.2f} on the object")

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display


def crop_box(mask, pad=0.12):
    ys, xs = np.nonzero(mask)
    p = int(pad * max(ys.max() - ys.min(), xs.max() - xs.min()))
    return (slice(max(ys.min() - p, 0), min(ys.max() + p, mask.shape[0])),
            slice(max(xs.min() - p, 0), min(xs.max() + p, mask.shape[1])))


picks = [views[i] for i in np.linspace(0, len(views) - 1, 4).round().astype(int)]
fig, axes = plt.subplots(len(picks), 4, figsize=(11, 3.4 * len(picks)))
for row, v in zip(axes, picks):
    box = crop_box(v["mask"])
    m = v["mask"][box][..., None]
    panels = [(v["name"], v["image"][box]), ("albedo", np.where(m, v["albedo"][box], 1.0)),
              ("roughness", np.where(m[..., 0], v["roughness"][box], np.nan)),
              ("metallic", np.where(m[..., 0], v["metallic"][box], np.nan))]
    for ax, (title, img) in zip(row, panels):
        ax.imshow(img, cmap="magma" if img.ndim == 2 else None, vmin=0, vmax=1)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=70, pil_kwargs={"quality": 85})
plt.close(fig)
display(IPImage(data=buf.getvalue()))

## Baking the views into textures

Every texel center that falls inside a chart sits on one point $P$ of the mesh, with normal $N$. A view with intrinsics $K$, rotation $R$ and translation $t$ sees $P$ at camera coordinates $X = R P + t$ and at the pixel

$$
x = f_x \frac{X_1}{X_3} + c_x, \qquad y = f_y \frac{X_2}{X_3} + c_y .
$$

A point on the far side of the object also projects to a pixel, and at that pixel the camera sees the near side. You therefore render the mesh's own depth map $D$ in each view and keep the sample only when $\lvert X_3 - D(x, y) \rvert$ is below a tolerance (2 mm), so that $P$ is the first surface along that pixel's ray. The sample must also fall inside the object's mask, eroded by two pixels so that pixels on the silhouette, which mix the object with the background, do not count.

The surviving samples are averaged with weights

$$
w = \max\!\left(0, \; N \cdot \frac{C - P}{\lVert C - P \rVert}\right),
$$

the cosine between the normal and the direction to the camera center $C = -R^\top t$. A view that sees the surface face-on resolves it finely, and a view at a grazing angle squeezes a large piece of surface into a few pixels, so the face-on view should count more. The texel's albedo, roughness and metallic are the weighted means of the views' maps at the pixel where it lands.

Some chart texels are seen by no view. Their values are copied from the nearest seen point on the surface, measured in 3D, and texels outside every chart copy their nearest chart texel, so that a renderer filtering across a chart's border never blends in black.

Write `bake`. The cell after it runs it twice: once on the Marigold-IID maps, and once with each photo in place of its albedo. The second texture keeps the lighting of your capture, and the cell compares the two.


In [ ]:
import numpy as np
import open3d as o3d
from scipy import ndimage
from scipy.spatial import cKDTree


def _scene(vertices, faces):
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.core.Tensor(np.asarray(vertices, np.float32)),
                        o3d.core.Tensor(np.asarray(faces, np.uint32)))
    return scene


def texel_surface(mesh_uv: dict, size: int) -> dict:
    """The surface point and normal behind every texel center of a size x size texture.

    Rasterizes the UV charts: a texel (row i, column j) has its center at
    uv = ((j + 0.5) / size, (i + 0.5) / size). Returns "chart" (size x size, bool: the texel
    center lies in a chart), and for the chart texels, in np.nonzero(chart) order, "points"
    (N x 3, meters) and "normals" (N x 3, unit), both interpolated from the triangle's vertices.
    """
    flat = np.c_[mesh_uv["uvs"] * size, np.zeros(len(mesh_uv["uvs"]))]
    scene = _scene(flat, mesh_uv["faces"])
    i, j = np.mgrid[0:size, 0:size]
    rays = np.zeros((size, size, 6), np.float32)
    rays[..., 0], rays[..., 1], rays[..., 2], rays[..., 5] = j + 0.5, i + 0.5, 1.0, -1.0
    hit = scene.cast_rays(o3d.core.Tensor(rays))
    chart = np.isfinite(hit["t_hit"].numpy())
    tri = mesh_uv["faces"][hit["primitive_ids"].numpy()[chart]]
    b = hit["primitive_uvs"].numpy()[chart]
    w = np.c_[1 - b.sum(1), b]  # barycentric weights of the triangle's three vertices
    points = np.einsum("nk,nkd->nd", w, mesh_uv["vertices"][tri])
    normals = np.einsum("nk,nkd->nd", w, mesh_uv["normals"][tri])
    return {"chart": chart, "points": points,
            "normals": normals / np.maximum(np.linalg.norm(normals, axis=1, keepdims=True), 1e-12)}


def depth_map(mesh_uv: dict, view: dict) -> np.ndarray:
    """The mesh's z-depth (meters) at every pixel center of a view; inf where no surface is hit."""
    h, w = view["mask"].shape
    K, R, t = view["K"], view["viewmat"][:3, :3], view["viewmat"][:3, 3]
    v, u = np.mgrid[0:h, 0:w]
    d = np.stack([(u + 0.5 - K[0, 2]) / K[0, 0], (v + 0.5 - K[1, 2]) / K[1, 1], np.ones((h, w))], -1) @ R
    rays = np.concatenate([np.broadcast_to(-R.T @ t, (h, w, 3)), d], -1).astype(np.float32)
    # the directions have unit z in the camera frame, so the hit distance is the z-depth
    return _scene(mesh_uv["vertices"], mesh_uv["faces"]).cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()


def fill_unseen(values: np.ndarray, covered: np.ndarray, surface: dict, size: int) -> dict:
    """Turn per-texel averages into full texture maps.

    values: N x C averages for the chart texels (np.nonzero(surface["chart"]) order), valid
    where covered (N, bool). A chart texel no view saw takes the value of the nearest covered
    texel on the surface, in 3D. A texel outside every chart takes the value of the nearest
    chart texel in the texture, so that filtering across a chart's border never blends in
    black. Returns "base_color" (size x size x 3), "roughness" and "metallic" (size x size),
    all float32 in [0, 1], and the masks "chart" and "covered" (size x size, bool).
    """
    if not covered.any():
        raise ValueError("no texel was seen by any view: check the poses, the masks and the depth test")
    filled = values.copy()
    _, nearest = cKDTree(surface["points"][covered]).query(surface["points"][~covered])
    filled[~covered] = values[covered][nearest]
    chart = surface["chart"]
    image = np.zeros((size, size, values.shape[1]))
    image[chart] = filled
    _, (ii, jj) = ndimage.distance_transform_edt(~chart, return_indices=True)
    image = np.clip(image[ii, jj], 0, 1).astype(np.float32)
    cov = np.zeros((size, size), bool)
    cov[chart] = covered
    return {"base_color": image[..., :3], "roughness": image[..., 3], "metallic": image[..., 4],
            "chart": chart, "covered": cov}

In [ ]:
import numpy as np


def bake(mesh_uv: dict, views: list[dict], size: int = 2048, depth_tol_m: float = 0.002,
         erode_px: int = 2) -> dict[str, np.ndarray]:
    """Bake the per-view Marigold-IID maps into size x size textures over the UV charts.

    views: dicts with "K", "viewmat" (world to camera), "mask" (H x W, bool) and the maps
    "albedo" (H x W x 3), "roughness" and "metallic" (H x W), all in [0, 1].

    1. surface = texel_surface(mesh_uv, size) gives the 3D point P and unit normal N behind
       every chart texel.
    2. For each view, project every P into the view: camera coordinates viewmat @ [P, 1],
       pixel x = fx X / Z + cx, y = fy Y / Z + cy, and the pixel (floor(x), floor(y)). Keep the
       texels in front of the camera that land inside the image, and of those only the ones
       where all of these hold:
       - the mask, eroded by erode_px pixels, is set at that pixel;
       - the texel is not occluded: its depth Z is within depth_tol_m of depth_map(mesh_uv, view)
         at that pixel (a texel hidden behind another part of the mesh is farther than the
         rendered depth there);
       - the surface faces the camera: cos = N . (C - P) / |C - P| > 0, with C the camera
         center -R^T t.
    3. Add cos times the view's (albedo, roughness, metallic) at that pixel to a running sum for
       the texel, and cos to its weight.
    4. A texel is covered if its weight is above 0. Divide the sums by the weights and return
       fill_unseen(values, covered, surface, size).
    """
    raise NotImplementedError

In [ ]:
t = time.time()
maps = bake(mesh_uv, views, size=2048)
bake_s = time.time() - t
chart, covered = maps["chart"], maps["covered"]
print(f"baked 2048 x 2048 maps in {bake_s:.0f} s: {chart.mean():.1%} of the texture is chart, "
      f"{covered.sum() / chart.sum():.1%} of the chart texels seen by at least one view")
print(f"mean over the seen texels: albedo {maps['base_color'][covered].mean(0).round(3)}, "
      f"roughness {maps['roughness'][covered].mean():.3f}, metallic {maps['metallic'][covered].mean():.3f}")

# The same bake with each photo in place of its albedo: a texture with the lighting left in.
photo_views = [dict(v, albedo=v["image"] / np.float32(255)) for v in views]
photo = bake(mesh_uv, photo_views, size=2048)["base_color"]
y_photo, y_albedo = luminance(photo[covered]), luminance(maps["base_color"][covered])
print(f"photo texture against albedo texture on the seen texels: luminance ratio median "
      f"{np.median(y_photo / np.maximum(y_albedo, 1e-3)):.2f}, mean absolute RGB difference "
      f"{np.abs(photo[covered] - maps['base_color'][covered]).mean():.3f}")

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

small = lambda a: cv2.resize(a.astype(np.float32), (512, 512), interpolation=cv2.INTER_AREA)
seen = np.where(covered, 1.0, np.where(chart, 0.45, 0.0))
fig, axes = plt.subplots(1, 5, figsize=(17, 3.9))
for ax, (title, img, cmap) in zip(axes, [("base color (albedo)", small(maps["base_color"]), None),
                                         ("photo texture (lighting kept)", small(photo), None),
                                         ("roughness", small(maps["roughness"]), "magma"),
                                         ("metallic", small(maps["metallic"]), "magma"),
                                         ("seen (white), unseen chart (gray)", small(seen), "gray")]):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=70, pil_kwargs={"quality": 85})
plt.close(fig)
display(IPImage(data=buf.getvalue()))

## Textured mesh in the viewer

The viewer shows the decimated mesh with the base color texture in the world frame of stage 1. The `bake_view` timeline steps through the 30 views: for each, the photo and the albedo Marigold-IID predicted from it, side by side. Scrub it and watch the same face of the object. The photo changes as the face turns toward or away from the light, and the albedo should change much less.


In [ ]:
def log_stage4(mesh_uv, maps, views):
    texture = (cv2.resize(maps["base_color"], (1024, 1024), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    rr.log("world/mesh_textured", rr.Mesh3D(vertex_positions=mesh_uv["vertices"], triangle_indices=mesh_uv["faces"],
                                            vertex_normals=mesh_uv["normals"], vertex_texcoords=mesh_uv["uvs"],
                                            albedo_texture=texture), static=True)
    for i, v in enumerate(views):
        rr.set_time("bake_view", sequence=i)
        k = 480 / v["image"].shape[1]
        for name, img in (("photo", v["image"]), ("albedo", (v["albedo"] * 255).astype(np.uint8))):
            img = cv2.resize(img, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
            rr.log(f"views/{name}", rr.EncodedImage(contents=cv2.imencode(".jpg", img[..., ::-1])[1].tobytes(),
                                                    media_type="image/jpeg"))


log_stage4(mesh_uv, maps, views)

In [ ]:
show_rerun()

## Check

All three maps must lie in $[0, 1]$, and at least 90% of the observable chart texels must be covered by at least one view. Two kinds of surface are not observable. Surface that faces down, with a normal whose $z$ component is below $-0.7$, lies under the object. The base of the mesh is the flat cap stage 3 closed on the support plane, which no camera sees because the object stands on it, and the underside of a part that clears the table by a few millimeters, such as the handle of a hammer lying flat, could be seen only from below the table's level. The rule depends on the mesh alone, so a face that looks down but that a camera could still see, an overhang high above the table, is left out as well. Surface whose outward normal ray meets the mesh again lies in a dent or crease. The surface around it blocks the view straight out, and any camera that still reaches it sees it at a grazing angle. The cell prints the share of chart texels in these two groups and the coverage of the rest.

On the reference object the check also compares the baked base color with the texture of the OmniObject3D scan, at the nearest scan point to each covered texel (millimeters, unit inferred). The scan's texture was photographed under the lighting of its capture rig, so it is a shaded texture, and the comparison is reported as a number with no pass threshold. The photo texture is compared with it the same way.


In [ ]:
import cv2
import numpy as np
from pathlib import Path


def write_stage4(out: Path, mesh_uv: dict, maps: dict) -> None:
    """Write stage4/ in one step: mesh_uv.obj (with mesh_uv.mtl) and the three maps as 8-bit PNGs.

    The OBJ stores texture coordinates with v up (OpenGL), so its vt is (u, 1 - v).
    """
    with atomic_dir(out) as d:
        lines = ["mtllib mesh_uv.mtl", "usemtl material"]
        lines += [f"v {x:.6f} {y:.6f} {z:.6f}" for x, y, z in mesh_uv["vertices"]]
        lines += [f"vt {u:.6f} {1 - v:.6f}" for u, v in mesh_uv["uvs"]]
        lines += [f"vn {x:.5f} {y:.5f} {z:.5f}" for x, y, z in mesh_uv["normals"]]
        lines += [f"f {a}/{a}/{a} {b}/{b}/{b} {c}/{c}/{c}" for a, b, c in mesh_uv["faces"] + 1]
        (d / "mesh_uv.obj").write_text("\n".join(lines) + "\n")
        (d / "mesh_uv.mtl").write_text("newmtl material\nKd 1 1 1\nmap_Kd base_color.png\n")
        to8 = lambda a: np.round(np.clip(a, 0, 1) * 255).astype(np.uint8)
        ok = [cv2.imwrite(str(d / "base_color.png"), to8(maps["base_color"])[..., ::-1]),
              cv2.imwrite(str(d / "roughness.png"), to8(maps["roughness"])),
              cv2.imwrite(str(d / "metallic.png"), to8(maps["metallic"]))]
        if not all(ok):
            raise RuntimeError("a texture could not be written; stage4/ was not written")

In [ ]:
import cv2
import numpy as np
import open3d as o3d
from pathlib import Path


def scan_albedo_difference(maps: dict, surface: dict, scan_path: Path, scan_from_world, max_gap_mm: float = 3.0,
                           n: int = 200_000, seed: int = 0) -> dict:
    """Mean absolute difference, per RGB channel in [0, 1], between the baked base color and the
    scan's texture at the nearest scan point, over seen texels within max_gap_mm of the scan.

    The scan (millimeters, unit inferred) is textured with its capture lighting baked in, so
    this is a comparison, not a ground truth for albedo.
    """
    scan = o3d.io.read_triangle_mesh(str(scan_path), True)
    texture = cv2.cvtColor(cv2.imread(str(Path(scan_path).with_name("Scan.jpg"))), cv2.COLOR_BGR2RGB) / 255.0
    corner_uvs = np.asarray(scan.triangle_uvs).reshape(-1, 3, 2)
    covered = maps["covered"][surface["chart"]]
    pick = np.random.default_rng(seed).permutation(np.nonzero(covered)[0])[:n]
    T = np.asarray(scan_from_world, dtype=np.float64)
    pts = surface["points"][pick] @ T[:3, :3].T + T[:3, 3]  # world meters to scan millimeters
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(scan))
    near = scene.compute_closest_points(o3d.core.Tensor(pts.astype(np.float32)))
    gap = np.linalg.norm(near["points"].numpy() - pts, axis=1)
    b = near["primitive_uvs"].numpy()
    uv = np.einsum("nk,nkd->nd", np.c_[1 - b.sum(1), b], corner_uvs[near["primitive_ids"].numpy()])
    h, w = texture.shape[:2]
    x = np.clip(uv[:, 0] * w - 0.5, 0, w - 1.001)  # OBJ texture coordinates have v up
    y = np.clip((1 - uv[:, 1]) * h - 0.5, 0, h - 1.001)
    x0, y0 = x.astype(int), y.astype(int)
    fx, fy = (x - x0)[:, None], (y - y0)[:, None]
    scan_rgb = ((1 - fy) * ((1 - fx) * texture[y0, x0] + fx * texture[y0, x0 + 1])
                + fy * ((1 - fx) * texture[y0 + 1, x0] + fx * texture[y0 + 1, x0 + 1]))
    ours = maps["base_color"][surface["chart"]][pick]
    close = gap < max_gap_mm
    return {"mean_abs_diff": float(np.abs(ours[close] - scan_rgb[close]).mean()), "texels": int(close.sum()),
            "ours": ours[close], "scan": scan_rgb[close], "pick": pick[close]}


def observable_texels(mesh_uv: dict, surface: dict, floor_z: float = 0.0) -> np.ndarray:
    """Which chart texels (np.nonzero(chart) order) lie on surface that a camera could see.

    Two kinds of surface are left out, both decided by the mesh alone. Surface that faces down
    (normal z below -0.7) is under the object: the flat cap on the support plane (within 2 mm of
    floor_z) rests on the table, and the underside of a part that clears the table by a few
    millimeters is seen, if at all, from below the table's level, where no camera is. Surface
    whose outward normal ray hits the mesh again lies in a dent or crease: the surface around it
    blocks the view straight out, and the views that remain see it at a grazing angle if at all.
    """
    P, N = surface["points"], surface["normals"]
    cap = (P[:, 2] < floor_z + 0.002) & (N[:, 2] < -0.7)
    down = N[:, 2] < -0.7  # the cap and every other face looking down
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.core.Tensor(np.asarray(mesh_uv["vertices"], np.float32)),
                        o3d.core.Tensor(np.asarray(mesh_uv["faces"], np.uint32)))
    rays = np.c_[P + 1e-4 * N, N].astype(np.float32)  # start a tenth of a millimeter off the surface
    inward = np.isfinite(scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy())
    return ~cap & ~down & ~inward


def stage4_problems(maps: dict, observable: np.ndarray, decimation_mm: float | None = None,
                    floor_mm: float | None = None, min_coverage: float = 0.9):
    """Return (failures, warnings, coverage). observable: chart texels a camera could see
    (observable_texels); coverage is the share of them that a view saw. decimation_mm is the
    Chamfer distance between the decimated and the full mesh, floor_mm the same measurement
    between two samplings of the full mesh; a decimation more than 3 times the floor warns."""
    failures, warnings = [], []
    if decimation_mm is not None and floor_mm is not None and decimation_mm > 3 * floor_mm:
        warnings.append(f"decimation moved the surface by {decimation_mm:.3f} mm, more than 3 times the sampling "
                        f"floor of {floor_mm:.3f} mm: raise target_triangles in unwrap so the textures stay on it")
    for k in ("base_color", "roughness", "metallic"):
        a = maps[k]
        if not np.isfinite(a).all() or a.min() < 0 or a.max() > 1:
            failures.append(f"{k} has values outside [0, 1]")
    seen = maps["covered"][maps["chart"]]
    coverage = float(seen[observable].mean()) if observable.any() else 0.0
    if coverage < min_coverage:
        failures.append(f"only {coverage:.1%} of the observable chart texels were seen by a view (at least "
                        f"{min_coverage:.0%} needed): check the poses, the masks and the occlusion test")
    if maps["metallic"][maps["covered"]].mean() > 0.5:
        warnings.append("the mean metallic value is above 0.5: check for strong highlights in the frames")
    return failures, warnings, coverage

In [ ]:
print(f"maps in [0, 1]: base color {maps['base_color'].min():.3f} to {maps['base_color'].max():.3f}, "
      f"roughness {maps['roughness'].min():.3f} to {maps['roughness'].max():.3f}, "
      f"metallic {maps['metallic'].min():.3f} to {maps['metallic'].max():.3f}")
surface = texel_surface(mesh_uv, 2048)
observable = observable_texels(mesh_uv, surface)
failures, warnings, coverage = stage4_problems(maps, observable, decimation_mm, floor_mm)
print(f"{1 - observable.mean():.1%} of the chart texels lie where no camera can see (facing down, the base on "
      f"the support plane among them, or in a dent the outward normal cannot leave); of the rest, {coverage:.1%} "
      f"were seen (at least 90% needed); of all chart texels, {maps['covered'][maps['chart']].mean():.1%}")
if REFERENCE:  # a comparison, not a gate: the scan's texture has its capture lighting baked in
    scale = json.loads((STAGE1 / "scale.json").read_text())
    scan_path = PROJECT / "ground_truth" / "scan" / "Scan.obj"
    scan_cmp = scan_albedo_difference(maps, surface, scan_path, scale["scan_from_world"])
    photo_cmp = scan_albedo_difference(dict(maps, base_color=photo), surface, scan_path, scale["scan_from_world"])
    print(f"against the scan texture over {scan_cmp['texels']:,} texels (not a gate): mean absolute difference "
          f"{scan_cmp['mean_abs_diff']:.3f} for the albedo, {photo_cmp['mean_abs_diff']:.3f} for the photo texture")
    print(f"mean RGB: scan texture {scan_cmp['scan'].mean(0).round(3)}, albedo {scan_cmp['ours'].mean(0).round(3)}, "
          f"photo texture {photo_cmp['ours'].mean(0).round(3)}")
print(f"stage time {(time.time() - T0) / 60:.1f} min")
for w in warnings:
    print("warning:", w)
assert not failures, "; ".join(failures)  # nothing is written when a gate fails
write_stage4(PROJECT / "stage4", mesh_uv, maps)
if REFERENCE:
    (PROJECT / "stage4" / ".reference").touch()  # stage 5 then knows it runs on the reference
print("passed")

### Question 1. Albedo or light?

Using $I = A \cdot S$, explain why one photo cannot tell a dark surface from a poorly lit one, and how Marigold-IID chooses one decomposition anyway. Describe what would show that the chosen albedo is wrong across views. Point to one face in the `bake_view` timeline where the photo's brightness changes and say what the albedo did there.

*Write your answer here.*

### Question 2. What the prior assumes

State what Marigold-IID's training data assumes about the scene, the light and the reflectance model, and where a close-up of a real object departs from it. Explain what the bake does when the views disagree about a material value. Point to a material map in the per-view figure that changes between views.

*Write your answer here.*

### Question 3. Which texels a view can see

Explain why the bake needs the depth test and the cosine weight, and what goes wrong without each. Then say which parts of a surface no outside view can observe, and why the coverage check counts only the observable texels. Point to the coverage line your check printed.

*Write your answer here.*